# Stage 1 Training — Cross-Embodiment Shared Latent Space

Yan et al. (2026) adaptation for dexterous hands.

**Setup checklist:**
- Runtime set to GPU: `Runtime > Change runtime type > T4 GPU`
- Repo synced to `MyDrive/AIST-hand/`
- `dex-urdf/` folder inside `MyDrive/AIST-hand/dex-urdf/`
- `hograspnet_abl11.csv` in `MyDrive/AIST-hand/` (root)
- `shadow_hand_right.yaml` in `MyDrive/AIST-hand/` (root)

## 0. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No GPU found. Go to Runtime > Change runtime type and select T4 GPU.')

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Config

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive/AIST-hand')  # Drive folder with data files
GITHUB_TOKEN = ''        # paste your GitHub token here (repo scope)
GITHUB_USER  = 'isapedraza'
REPO_NAME    = 'AIST-hand'

# Training hyperparameters
B          = 20_000    # batch size (reduced from Yan 1e5 for T4 memory)
N_STEPS    = 5_000     # training steps
LOG_EVERY  = 50        # print losses every N steps
CKPT_EVERY = 500       # save checkpoint every N steps
LR_WARMUP  = 500       # steps before cosine LR decay begins

print(f'Drive root: {DRIVE_ROOT}')
print(f'B={B}, N_STEPS={N_STEPS}')

## 3. Clone repo from GitHub

In [ ]:
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    clone_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
    os.system(f'git clone {clone_url} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull')
    print('Already cloned, pulled latest.')

REPO_ROOT = Path(REPO_DIR)
DEX_ROOT  = DRIVE_ROOT / 'dex-urdf'
print(f'Repo: {REPO_ROOT}')
print(f'dex-urdf: {DEX_ROOT}')

## 4. Install dependencies

In [ ]:
import torch
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION  = torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f'torch={TORCH_VERSION}, cuda={CUDA_VERSION}')

!pip install -q torch-geometric
!pip install -q pytorch-kinematics
!pip install -q -e /content/AIST-hand/grasp-model

print('Done.')

## 5. Run Stage 1 training

In [ ]:
import torch

CSV_PATH    = DRIVE_ROOT / 'hograspnet_abl11.csv'
URDF_PATH   = DEX_ROOT  / 'robots/hands/shadow_hand/shadow_hand_right.urdf'
HAND_CONFIG = DRIVE_ROOT / 'shadow_hand_right.yaml'
CKPT_PATH   = DRIVE_ROOT / 'checkpoints/stage1_latest.pt'

Z_DIM      = 16
SHARED_DIM = 1024
LR         = 1e-3
LAMBDA_C   = 10.0
LAMBDA_REC = 5.0
LAMBDA_LTC = 1.0
LAMBDA_TMP = 0.1
N_TRIPLETS = 2048

print(f'CSV : {CSV_PATH}')
print(f'CKPT: {CKPT_PATH}')

In [ ]:
import subprocess, sys

cmd = [
    sys.executable,
    str(REPO_ROOT / "grasp-model/src/cross_emb/train_cross_emb.py"),
    "--repo_root",   str(REPO_ROOT),
    "--dex_root",    str(DEX_ROOT),
    "--csv_path",    str(CSV_PATH),
    "--ckpt_path",   str(CKPT_PATH),
    "--hand_config", str(HAND_CONFIG),
    "--b",           str(B),
    "--n_steps",     str(N_STEPS),
    "--log_every",   str(LOG_EVERY),
    "--ckpt_every",  str(CKPT_EVERY),
    "--lr_warmup",   str(LR_WARMUP),
    "--z_dim",       str(Z_DIM),
    "--shared_dim",  str(SHARED_DIM),
    "--lr",          str(LR),
    "--lambda_c",    str(LAMBDA_C),
    "--lambda_rec",  str(LAMBDA_REC),
    "--lambda_ltc",  str(LAMBDA_LTC),
    "--lambda_tmp",  str(LAMBDA_TMP),
    "--n_triplets",  str(N_TRIPLETS),
]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"Training failed with code {proc.returncode}")

In [ ]:
# GPU memory diagnostics -- run after first step to check headroom
import torch
allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
total     = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"Allocated : {allocated:.2f} GB")
print(f"Reserved  : {reserved:.2f} GB")
print(f"Total     : {total:.2f} GB")
print(f"Free      : {total - reserved:.2f} GB")